In [2]:
# ==============================================================================
# 重要度比較スクリプト: Coef (|beta|) vs Permutation Importance
# ターゲット: Jsc | データ: m_all_OH_rebuilt.csv
# ==============================================================================

suppressPackageStartupMessages({
  library(dplyr)
  library(ggplot2)
  library(tidyr)
})

# -------------------------------
# 1. パス設定
# -------------------------------
# 入力ソース
path_coef <- "/Volumes/csbdeep11/_yasu-i/20250303_rebuiled/after_20251225/20260128_Importance_GPR_Linear/Corr_1000_GPR_Linear_Coef/importance_GPR_Linear_Coef.csv"
path_perm <- "/Volumes/csbdeep11/_yasu-i/20250303_rebuiled/after_20251225/Corr_1000/results/Corr_1000_GPR_Linear/all_importance_GPR_Linear.csv"

# 出力先
out_dir <- "/Volumes/csbdeep11/_yasu-i/20250303_rebuiled/after_20251225/20260128_Importance_GPR_Linear"
if (!dir.exists(out_dir)) dir.create(out_dir, recursive = TRUE)

# -------------------------------
# 2. データ読み込みとフィルタリング
# -------------------------------
# 抽出条件
target_file <- "m_all_OH_rebuilt.csv"
target_var  <- "Jsc"

# 係数ベース重要度の読み込み
df_coef <- read.csv(path_coef) %>%
  filter(File == target_file, Target == target_var) %>%
  select(Feature, Coef_Abs = Importance)

# 置換重要度の読み込み
df_perm <- read.csv(path_perm) %>%
  filter(File == target_file, Target == target_var) %>%
  select(Feature, Perm_Imp = Importance)

# -------------------------------
# 3. データの結合と正規化
# -------------------------------
# 二つの指標を結合（両方に存在する変数のみ）
comparison_df <- inner_join(df_coef, df_perm, by = "Feature")

# 比較しやすくするため、それぞれ最大値で正規化 (0-1)
comparison_df <- comparison_df %>%
  mutate(
    Coef_Norm = Coef_Abs / max(Coef_Abs, na.rm = TRUE),
    Perm_Norm = Perm_Imp / max(Perm_Imp, na.rm = TRUE)
  ) %>%
  arrange(desc(Perm_Norm))

# -------------------------------
# 4. 結果の出力
# -------------------------------
# ① CSV出力
write.csv(comparison_df, file.path(out_dir, "GPR_Importance_Comparison_Jsc_m_all_OH.csv"), row.names = FALSE)

# ② 可視化：散布図による相関確認
p_scatter <- ggplot(comparison_df, aes(x = Coef_Norm, y = Perm_Norm)) +
  geom_point(alpha = 0.6, color = "blue") +
  geom_text(aes(label = ifelse(Perm_Norm > 0.5 | Coef_Norm > 0.5, Feature, "")), 
            vjust = -1, size = 3, check_overlap = TRUE) +
  geom_abline(slope = 1, intercept = 0, linetype = "dashed", color = "red") +
  labs(title = paste("Importance Correlation: Jsc in", target_file),
       x = "Normalized Coefficient Absolute (|beta|)",
       y = "Normalized Permutation Importance (R2 drop)") +
  theme_minimal()

ggsave(file.path(out_dir, "GPR_Importance_Correlation_Plot.png"), p_scatter, width = 8, height = 6)

# ③ 可視化：上位変数の比較バーチャート
p_bar <- comparison_df %>%
  slice_max(order_by = Perm_Norm, n = 20) %>%
  pivot_longer(cols = c(Coef_Norm, Perm_Norm), names_to = "Method", values_to = "Value") %>%
  ggplot(aes(x = reorder(Feature, Value), y = Value, fill = Method)) +
  geom_bar(stat = "identity", position = "dodge") +
  coord_flip() +
  labs(title = "Top 20 Features Comparison", x = "Features", y = "Normalized Importance") +
  scale_fill_manual(values = c("Coef_Norm" = "steelblue", "Perm_Norm" = "coral")) +
  theme_minimal()

ggsave(file.path(out_dir, "GPR_Top20_Comparison_Bar.png"), p_bar, width = 8, height = 8)

cat("\n[DONE] Comparison analysis completed.\n")
cat("Outputs saved in:", out_dir, "\n")


[DONE] Comparison analysis completed.
Outputs saved in: /Volumes/csbdeep11/_yasu-i/20250303_rebuiled/after_20251225/20260128_Importance_GPR_Linear 


In [3]:
# ==============================================================================
# 重要度比較スクリプト（コンソール出力版）
# ターゲット: Jsc | データ: m_all_OH_rebuilt.csv
# ==============================================================================

suppressPackageStartupMessages({
  library(dplyr)
  library(tidyr)
})

# -------------------------------
# 1. パス設定
# -------------------------------
path_coef <- "/Volumes/csbdeep11/_yasu-i/20250303_rebuiled/after_20251225/20260128_Importance_GPR_Linear/Corr_1000_GPR_Linear_Coef/importance_GPR_Linear_Coef.csv"
path_perm <- "/Volumes/csbdeep11/_yasu-i/20250303_rebuiled/after_20251225/Corr_1000/results/Corr_1000_GPR_Linear/all_importance_GPR_Linear.csv"

# -------------------------------
# 2. データ読み込みとフィルタリング
# -------------------------------
target_file <- "m_all_OH_rebuilt.csv"
target_var  <- "Jsc"

# ファイルの存在確認
if (!file.exists(path_coef)) stop("Coefficient importance file not found.")
if (!file.exists(path_perm)) stop("Permutation importance file not found.")

df_coef <- read.csv(path_coef) %>%
  filter(File == target_file, Target == target_var) %>%
  select(Feature, Coef_Abs = Importance)

df_perm <- read.csv(path_perm) %>%
  filter(File == target_file, Target == target_var) %>%
  select(Feature, Perm_Imp = Importance)

# -------------------------------
# 3. データの結合と正規化
# -------------------------------
comparison_df <- inner_join(df_coef, df_perm, by = "Feature")

# 最大値で正規化 (0-1) して比較しやすくする
comparison_df <- comparison_df %>%
  mutate(
    Coef_Norm = Coef_Abs / max(Coef_Abs, na.rm = TRUE),
    Perm_Norm = Perm_Imp / max(Perm_Imp, na.rm = TRUE)
  ) %>%
  arrange(desc(Perm_Norm)) # Permutation重要度の高い順に並び替え

# -------------------------------
# 4. コンソールへの結果表示
# -------------------------------
cat("\n==========================================================\n")
cat(" Importance Comparison: Jsc (m_all_OH_rebuilt.csv)\n")
cat("==========================================================\n")
cat(sprintf("%-30s | %-10s | %-10s\n", "Feature", "Coef_Norm", "Perm_Norm"))
cat("----------------------------------------------------------\n")

# 上位20位を表示
top_n <- head(comparison_df, 20)
for(i in 1:nrow(top_n)) {
  cat(sprintf("%-30s | %-10.4f | %-10.4f\n", 
              top_n$Feature[i], 
              top_n$Coef_Norm[i], 
              top_n$Perm_Norm[i]))
}

# 相関係数の算出
cor_val <- cor(comparison_df$Coef_Norm, comparison_df$Perm_Norm, use="complete.obs")
cat("----------------------------------------------------------\n")
cat(sprintf("Correlation between Coef and Perm: %.4f\n", cor_val))
cat("==========================================================\n\n")

# ファイル保存（バックアップ用）
out_dir <- "/Volumes/csbdeep11/_yasu-i/20250303_rebuiled/after_20251225/20260128_Importance_GPR_Linear"
if (!dir.exists(out_dir)) dir.create(out_dir, recursive = TRUE)
write.csv(comparison_df, file.path(out_dir, "GPR_Importance_Comparison_Console_Output.csv"), row.names = FALSE)


 Importance Comparison: Jsc (m_all_OH_rebuilt.csv)
Feature                        | Coef_Norm  | Perm_Norm 
----------------------------------------------------------
SMILESsnamenM_PC61BM           | 1.0000     | 1.0000    
SMILESsnamep1M_DTA             | 0.2974     | 0.7450    
SMILESsnamenM_PC71BM           | 0.6384     | 0.3735    
SMILESsnamep1M_PhBADT          | 0.1257     | 0.3372    
SMILESsnamep1M_PNTz4T          | 0.1598     | 0.2713    
Lay5electronodes1_             | 0.3716     | 0.2594    
Lay5electronodes1_Ca           | 0.4237     | 0.2035    
SMILESsnamep1M_PBDTFQT         | 0.1232     | 0.1992    
Mmonomerp1M                    | 0.1202     | 0.1925    
SMILESsnamep1M_AtD             | 0.0831     | 0.1912    
Lay6electronodes2_             | 0.3263     | 0.1902    
SMILESsnamep1M_BTIBDTP2        | 0.0808     | 0.1746    
SMILESsnamep1M_EHDBTA          | 0.0567     | 0.1507    
SMILESsnamenM_PTBI2T           | 0.0303     | 0.1451    
SMILESsnamep1M_C10DPPBP        | 0